# Distribution, Dominance, and Simulation Workflow


In [1]:
import numpy as np

from pynns import (
    fsd,
    nns_anova,
    nns_cdf,
    nns_mc,
    nns_meboot,
    nns_norm,
    nns_sd_cluster,
    nns_ss,
    sd_efficient_set,
    ssd,
    tsd,
)

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(99)


## Return distributions


In [2]:
n = 180
defensive = rng.normal(0.00045, 0.0060, n)
balanced = rng.normal(0.00065, 0.0080, n)
aggressive = rng.normal(0.00095, 0.0130, n)
aggressive[::29] -= 0.045
balanced[::47] -= 0.020
returns = np.column_stack((defensive, balanced, aggressive))
names = ("defensive", "balanced", "aggressive")

print("name        mean      vol       min       max")
for i, name in enumerate(names):
    x = returns[:, i]
    print(f"{name:<10} {np.mean(x):>8.5f} {np.std(x):>8.5f} {np.min(x):>8.5f} {np.max(x):>8.5f}")


name        mean      vol       min       max
defensive   0.00069  0.00580 -0.01641  0.01615
balanced    0.00009  0.00914 -0.02656  0.02474
aggressive -0.00010  0.01596 -0.08513  0.03241


## CDF, survival, and hazard


In [3]:
cdf = nns_cdf(aggressive, degree=0, target=0.0)
survival = nns_cdf(aggressive, degree=0, type="survival")
hazard = nns_cdf(aggressive, degree=0, type="cumulative hazard", target=0.0)

fn = cdf["Function"]
sfn = survival["Function"]
print("first five aggressive CDF/survival rows:")
print(np.column_stack((fn["x"][:5], fn["CDF"][:5], sfn["S(x)"][:5])))
print("CDF at 0:", cdf["target.value"])
print("cumulative hazard at 0:", hazard["target.value"])


first five aggressive CDF/survival rows:
[[-0.0851  0.0056  0.9944]
 [-0.0595  0.0111  0.9889]
 [-0.0373  0.0167  0.9833]
 [-0.0349  0.0222  0.9778]
 [-0.0335  0.0278  0.9722]]
CDF at 0: [0.4778]
cumulative hazard at 0: [0.6529]


## ANOVA-style comparison


In [4]:
pairwise = nns_anova([defensive, balanced, aggressive], pairwise=True, confidence_interval=None)
robust = nns_anova(defensive, aggressive, robust=True, n_boot=128, random_seed=101, confidence_interval=None)
print("pairwise certainty matrix:")
print(pairwise)
print("defensive vs aggressive robust certainty:", round(float(robust["Certainty"]), 4))


pairwise certainty matrix:
[[1.     0.6674 0.4353]
 [0.6674 1.     0.6606]
 [0.4353 0.6606 1.    ]]
defensive vs aggressive robust certainty: 0.4353


## Stochastic dominance


In [5]:
print("pairwise dominance result codes: 1 means first dominates, -1 means second dominates, 0 means neither")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        print(
            f"{names[i]} vs {names[j]}:"
            f" FSD={fsd(returns[:, i], returns[:, j])},"
            f" SSD={ssd(returns[:, i], returns[:, j])},"
            f" TSD={tsd(returns[:, i], returns[:, j])}"
        )

efficient = sd_efficient_set(returns, degree=2)
clusters = nns_sd_cluster(returns, degree=2, names=names, min_cluster=1)
print("degree-2 efficient set names:", [names[index] for index in efficient])
print("degree-2 SD clusters:", clusters["Clusters"])


pairwise dominance result codes: 1 means first dominates, -1 means second dominates, 0 means neither
defensive vs balanced: FSD=0, SSD=1, TSD=1
defensive vs aggressive: FSD=0, SSD=1, TSD=1
balanced vs aggressive: FSD=0, SSD=1, TSD=1
degree-2 efficient set names: ['defensive']
degree-2 SD clusters: {'Cluster_1': ['defensive'], 'Cluster_2': ['balanced'], 'Cluster_3': ['aggressive']}


## Stochastic superiority


In [6]:
print("x vs y stochastic superiority: p_gt, p_tie, p_star")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        ss = nns_ss(returns[:, i], returns[:, j])
        print(f"{names[i]} > {names[j]}:", {key: round(value, 4) for key, value in ss.items()})


x vs y stochastic superiority: p_gt, p_tie, p_star
defensive > balanced: {'p_gt': 0.5175, 'p_tie': 0.0, 'p_star': 0.5175}
defensive > aggressive: {'p_gt': 0.4899, 'p_tie': 0.0, 'p_star': 0.4899}
balanced > aggressive: {'p_gt': 0.4806, 'p_tie': 0.0, 'p_star': 0.4806}


## Bootstrap and Monte Carlo


In [7]:
meboot = nns_meboot(balanced[:60], reps=5, rho=0.25, random_seed=202)
mc = nns_mc(balanced[:60], reps=4, lower_rho=-0.5, upper_rho=0.5, by=0.5, random_seed=303)
print("meboot replicate matrix shape:", meboot["replicates"].shape)
print("meboot ensemble head:", np.round(meboot["ensemble"][:6], 5))
print("mc rho labels:", list(mc["replicates"].keys()))
print("mc ensemble head:", np.round(mc["ensemble"][:6], 5))


meboot replicate matrix shape: (60, 5)
meboot ensemble head: [-0.0122 -0.0081 -0.0004 -0.0069 -0.0013 -0.0023]
mc rho labels: ['rho = 0.5', 'rho = 0', 'rho = -0.5']
mc ensemble head: [-0.0132 -0.0032 -0.0024 -0.0021 -0.001  -0.0044]


## Normalization


In [8]:
macro_panel = np.column_stack(
    (
        100.0 + np.cumsum(returns[:, 0]),
        5000.0 + 50.0 * np.cumsum(returns[:, 1]),
        2.0 + np.cumsum(returns[:, 2]),
    )
)
normalized = nns_norm(macro_panel, linear=False)
print("original means:", np.round(np.mean(macro_panel, axis=0), 3))
print("normalized means:", np.round(np.mean(normalized, axis=0), 3))
print("normalized first row:", np.round(normalized[0], 3))


original means: [ 100.079 5000.378    1.9  ]
normalized means: [1211.59  1690.712  914.769]
normalized first row: [1210.642 1690.135  922.033]
